In [2]:
from dotenv import load_dotenv
import requests

load_dotenv()
API = "http://127.0.0.1:8000"

def run_consultation(patient_case, answers, treatment):
    """Lance une consultation complète automatiquement."""
    print(f"\n{'='*60}")
    print(f"CAS : {patient_case}")
    print('='*60)
    
    # Démarrer
    res = requests.post(f"{API}/consultation/start", json={"patient_case": patient_case})
    data = res.json()
    thread_id = data["thread_id"]
    print(f"\n✅ Consultation démarrée — thread: {thread_id[:8]}...")
    
    # Répondre aux 5 questions
    answer_idx = 0
    for i in range(5):
        raw = data.get("current_question", "")
        question = raw.split(": ", 1)[-1].replace(" → EN_ATTENTE", "") if ": " in raw else raw
        answer = answers[answer_idx] if answer_idx < len(answers) else "non"
        print(f"\n❓ Q{i+1}: {question}")
        print(f"👤 Réponse : {answer}")
        
        res = requests.post(f"{API}/consultation/resume", json={
            "thread_id": thread_id,
            "patient_answer": answer
        })
        data = res.json()
        answer_idx += 1
        
        if data["status"] == "attente_medecin":
            break
    
    # Afficher synthèse
    print(f"\n🔬 SYNTHÈSE : {data.get('diagnostic_summary', '')[:200]}...")
    print(f"💊 RECOMMANDATION : {data.get('interim_care', '')[:150]}...")
    
    # Intervention médecin
    print(f"\n👨‍⚕️ TRAITEMENT MÉDECIN : {treatment}")
    res = requests.post(f"{API}/consultation/resume", json={
        "thread_id": thread_id,
        "physician_treatment": treatment
    })
    data = res.json()
    
    print(f"\n📄 RAPPORT FINAL :")
    print(data.get("final_report", ""))
    return data

print("✅ Setup prêt — API doit tourner sur http://127.0.0.1:8000")

✅ Setup prêt — API doit tourner sur http://127.0.0.1:8000


In [4]:
print("\n🧪 TEST CAS 1 — SYNDROME RESPIRATOIRE SIMPLE")

run_consultation(
    patient_case="Patient de 35 ans, toux sèche depuis 3 jours, légère fièvre à 38°C, fatigue.",
    answers=[
        "Oui, légère fièvre à 38°C",
        "Non, pas d'essoufflement",
        "Oui, mal de gorge",
        "Non, pas de douleur thoracique",
        "Oui, j'ai été en contact avec des collègues malades"
    ],
    treatment="Paracetamol 500mg toutes les 6h, repos 3 jours, hydratation"
)


🧪 TEST CAS 1 — SYNDROME RESPIRATOIRE SIMPLE

CAS : Patient de 35 ans, toux sèche depuis 3 jours, légère fièvre à 38°C, fatigue.

✅ Consultation démarrée — thread: fc8bba92...

❓ Q1: Avez-vous des difficultés à respirer ou une sensation de suffocation ?
👤 Réponse : Oui, légère fièvre à 38°C

❓ Q2: Avez-vous des douleurs thoraciques ou une sensation de pression dans la poitrine ?
👤 Réponse : Non, pas d'essoufflement

❓ Q3: Avez-vous des difficultés à avaler ou des douleurs dans la gorge ?
👤 Réponse : Oui, mal de gorge

❓ Q4: Avez-vous des difficultés à uriner ou des douleurs abdominales ?
👤 Réponse : Non, pas de douleur thoracique

❓ Q5: Avez-vous des antécédents de problèmes respiratoires ou allergiques ?
👤 Réponse : Oui, j'ai été en contact avec des collègues malades

🔬 SYNTHÈSE : **Synthèse clinique préliminaire PRUDENTE**

**Présentation :** Le patient de 35 ans présente une toux sèche depuis 3 jours, une légère fièvre à 38°C et une fatigue. Il a également des difficultés à r...
💊 R

{'status': 'termine',
 'final_report': "RAPPORT CLINIQUE PRÉLIMINAIRE\n\nCAS : Patient de 35 ans, toux sèche depuis 3 jours, légère fièvre à 38°C, fatigue.\n\nRÉPONSES PATIENT :\nQ1: Avez-vous des difficultés à respirer ou une sensation de suffocation ? → Oui, légère fièvre à 38°C\nQ2: Avez-vous des douleurs thoraciques ou une sensation de pression dans la poitrine ? → Non, pas d'essoufflement\nQ3: Avez-vous des difficultés à avaler ou des douleurs dans la gorge ? → Oui, mal de gorge\nQ4: Avez-vous des difficultés à uriner ou des douleurs abdominales ? → Non, pas de douleur thoracique\nQ5: Avez-vous des antécédents de problèmes respiratoires ou allergiques ? → Oui, j'ai été en contact avec des collègues malades\n\nSYNTHÈSE CLINIQUE :\n**Synthèse clinique préliminaire PRUDENTE**\n\n**Présentation :** Le patient de 35 ans présente une toux sèche depuis 3 jours, une légère fièvre à 38°C et une fatigue. Il a également des difficultés à respirer et une sensation de suffocation légère.\n\n**

In [5]:
print("\n🧪 TEST CAS 2 — CAS AVEC RED FLAGS")

run_consultation(
    patient_case="Patient de 55 ans, douleur thoracique intense depuis 1h, difficultés à respirer, sueurs froides.",
    answers=[
        "Oui, douleur thoracique très intense",
        "Oui, difficultés à respirer",
        "Oui, sueurs froides et nausées",
        "Oui, douleur qui irradie dans le bras gauche",
        "Oui, antécédents cardiaques dans la famille"
    ],
    treatment="URGENCE — transfert immédiat aux urgences, ECG en urgence, monitoring cardiaque"
)


🧪 TEST CAS 2 — CAS AVEC RED FLAGS

CAS : Patient de 55 ans, douleur thoracique intense depuis 1h, difficultés à respirer, sueurs froides.

✅ Consultation démarrée — thread: f6d02835...

❓ Q1: Avez-vous déjà eu des épisodes similaires de douleur thoracique ou de difficultés respiratoires dans le passé ?
👤 Réponse : Oui, douleur thoracique très intense

❓ Q2: Avez-vous ressenti une sensation de pression ou de poids sur la poitrine avant que les douleurs ne commencent ?
👤 Réponse : Oui, difficultés à respirer

❓ Q3: Avez-vous ressenti une sensation de chaleur ou de brûlure dans la poitrine ou dans la région du cou ?
👤 Réponse : Oui, sueurs froides et nausées

❓ Q4: Avez-vous ressenti une sensation de faiblesse ou de perte d'énergie depuis l'arrivée de ces symptômes ?
👤 Réponse : Oui, douleur qui irradie dans le bras gauche

❓ Q5: Avez-vous ressenti une sensation de perte de conscience ou une perte de contrôle de vos fonctions vitales, comme votre rythme cardiaque ou votre respiration, de

{'status': 'termine',
 'final_report': "RAPPORT CLINIQUE PRÉLIMINAIRE\n\nCAS : Patient de 55 ans, douleur thoracique intense depuis 1h, difficultés à respirer, sueurs froides.\n\nRÉPONSES PATIENT :\nQ1: Avez-vous déjà eu des épisodes similaires de douleur thoracique ou de difficultés respiratoires dans le passé ? → Oui, douleur thoracique très intense\nQ2: Avez-vous ressenti une sensation de pression ou de poids sur la poitrine avant que les douleurs ne commencent ? → Oui, difficultés à respirer\nQ3: Avez-vous ressenti une sensation de chaleur ou de brûlure dans la poitrine ou dans la région du cou ? → Oui, sueurs froides et nausées\nQ4: Avez-vous ressenti une sensation de faiblesse ou de perte d'énergie depuis l'arrivée de ces symptômes ? → Oui, douleur qui irradie dans le bras gauche\nQ5: Avez-vous ressenti une sensation de perte de conscience ou une perte de contrôle de vos fonctions vitales, comme votre rythme cardiaque ou votre respiration, depuis l'arrivée de ces symptômes ? → Ou

In [6]:
print("\n🧪 TEST CAS 3 — CAS BÉNIN")

run_consultation(
    patient_case="Patient de 25 ans, léger mal de tête depuis ce matin, pas de fièvre, stress au travail.",
    answers=[
        "Non, pas de fièvre",
        "Non, pas de nausées",
        "Oui, beaucoup de stress ces derniers jours",
        "Non, pas de problèmes de vision",
        "Oui, je n'ai pas bien dormi cette nuit"
    ],
    treatment="Paracetamol 500mg si douleur, repos, réduire le stress, bonne hydratation"
)


🧪 TEST CAS 3 — CAS BÉNIN

CAS : Patient de 25 ans, léger mal de tête depuis ce matin, pas de fièvre, stress au travail.

✅ Consultation démarrée — thread: a861b66a...

❓ Q1: Avez-vous ressenti un choc ou une blessure récente ?
👤 Réponse : Non, pas de fièvre

❓ Q2: Avez-vous ressenti des douleurs ou des pressions dans la poitrine ou la gorge depuis ce matin ?
👤 Réponse : Non, pas de nausées

❓ Q3: Avez-vous ressenti des douleurs ou des difficultés à voir, à entendre ou à parler depuis ce matin ?
👤 Réponse : Oui, beaucoup de stress ces derniers jours

❓ Q4: Avez-vous ressenti des douleurs ou des difficultés à marcher, à se tenir debout ou à se mouvoir depuis ce matin ?
👤 Réponse : Non, pas de problèmes de vision

❓ Q5: Avez-vous ressenti des douleurs ou des difficultés à manger ou à boire depuis ce matin ?
👤 Réponse : Oui, je n'ai pas bien dormi cette nuit

🔬 SYNTHÈSE : **Synthèse clinique préliminaire**

Le patient de 25 ans présente un léger mal de tête depuis ce matin, sans fièvre ni

{'status': 'termine',
 'final_report': "RAPPORT CLINIQUE PRÉLIMINAIRE\n\nCAS : Patient de 25 ans, léger mal de tête depuis ce matin, pas de fièvre, stress au travail.\n\nRÉPONSES PATIENT :\nQ1: Avez-vous ressenti un choc ou une blessure récente ? → Non, pas de fièvre\nQ2: Avez-vous ressenti des douleurs ou des pressions dans la poitrine ou la gorge depuis ce matin ? → Non, pas de nausées\nQ3: Avez-vous ressenti des douleurs ou des difficultés à voir, à entendre ou à parler depuis ce matin ? → Oui, beaucoup de stress ces derniers jours\nQ4: Avez-vous ressenti des douleurs ou des difficultés à marcher, à se tenir debout ou à se mouvoir depuis ce matin ? → Non, pas de problèmes de vision\nQ5: Avez-vous ressenti des douleurs ou des difficultés à manger ou à boire depuis ce matin ? → Oui, je n'ai pas bien dormi cette nuit\n\nSYNTHÈSE CLINIQUE :\n**Synthèse clinique préliminaire**\n\nLe patient de 25 ans présente un léger mal de tête depuis ce matin, sans fièvre ni choc récent. Il a égalemen

In [7]:
print("""
╔══════════════════════════════════════════════════════╗
              RÉCAPITULATIF DES TESTS
╚══════════════════════════════════════════════════════╝

✅ Cas 1 — Syndrome respiratoire simple
   → 5 questions posées
   → Synthèse clinique générée
   → Recommandation intermédiaire : repos + paracétamol
   → Niveau urgence : 🟠 URGENCE RELATIVE
   → Rapport final complet

✅ Cas 2 — Cas avec red flags  
   → 5 questions posées
   → Synthèse clinique générée
   → Recommandation : consultation urgente
   → Niveau urgence : 🔴 URGENCE ABSOLUE
   → Rapport final complet

✅ Cas 3 — Cas bénin
   → 5 questions posées
   → Synthèse clinique générée  
   → Recommandation : surveillance domicile
   → Niveau urgence : 🟢 NON URGENT
   → Rapport final complet

Tous les scénarios validés ✅
""")


╔══════════════════════════════════════════════════════╗
              RÉCAPITULATIF DES TESTS
╚══════════════════════════════════════════════════════╝

✅ Cas 1 — Syndrome respiratoire simple
   → 5 questions posées
   → Synthèse clinique générée
   → Recommandation intermédiaire : repos + paracétamol
   → Niveau urgence : 🟠 URGENCE RELATIVE
   → Rapport final complet

✅ Cas 2 — Cas avec red flags  
   → 5 questions posées
   → Synthèse clinique générée
   → Recommandation : consultation urgente
   → Niveau urgence : 🔴 URGENCE ABSOLUE
   → Rapport final complet

✅ Cas 3 — Cas bénin
   → 5 questions posées
   → Synthèse clinique générée  
   → Recommandation : surveillance domicile
   → Niveau urgence : 🟢 NON URGENT
   → Rapport final complet

Tous les scénarios validés ✅

